# E-Commerce Review Sentiment Analysis (DistilBERT)

Production-style NLP pipeline for classifying product reviews into **negative** and **positive** sentiment.

## Why this project matters
- Demonstrates transfer learning with Hugging Face Transformers.
- Handles class imbalance using weighted loss.
- Includes reproducible training, structured evaluation, and model export for inference.

## Project workflow
1. Load and validate review data.
2. Preprocess text and create sentiment labels.
3. Tokenize and build Hugging Face datasets.
4. Fine-tune DistilBERT with weighted cross-entropy.
5. Evaluate with accuracy, F1-score, classification report, and confusion matrix.
6. Save model artifacts for deployment.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

In [ ]:
# Prefer a local data/ folder for portability.
DATA_PATH = Path("data/7817_1.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

df.sample(5, random_state=SEED)

Dataset shape: (1597, 27)


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
802,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/kindle-fire-hdx-student-...,I love this handheld device especially all the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,AVpfpzCi1cnluZ0-oxEr,B00DOPNLJ0,Amazon,Amazon Devices,NaN,2015-06-02T08:44:19Z,2017-08-07T15:41:59Z,NaN,NaN,kindlefirehdx89/b00dopnlj0,...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,NaN,NaN,B. Tarbuck,NaN,NaN,NaN
350,AVpfLiCSilAPnD_xWpk_,B00CX5P8FC,Amazon,"Categories,Amazon Devices,Electronics Features...",NaN,2015-05-22T18:12:20Z,2017-08-08T22:03:26Z,NaN,8.487190e+11,"848719022827,0848719022827,amazonfiretv/b00cx5...",...,NaN,http://www.amazon.com/Fire-TV-streaming-media-...,This was easy to set up I can access many movi...,Amazon Fire TV - A must have!!!!,NaN,NaN,Amazon Customer,NaN,8.487190e+11,NaN
682,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,5.0,https://www.amazon.com/Amazon-Echo-Case-fits-G...,I am thoroughly impressed with my Echo Dot and...,The Extra Touch,NaN,NaN,dm,NaN,NaN,NaN
1324,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great little device easy to connect to bluetoo...,Easy small decent sound,NaN,NaN,Drjim,NaN,8.416670e+11,1.75 lbs


## 1. Text Preprocessing and Label Engineering

In [ ]:
required_cols = ["reviews.text", "reviews.rating"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy()
df["reviews.text"] = df["reviews.text"].astype(str).str.strip()
df = df[df["reviews.text"].str.len() > 0]
df = df.drop_duplicates(subset=["reviews.text"])

def label_sentiment(rating: float) -> int:
    return 1 if rating >= 4 else 0

df["label"] = df["reviews.rating"].apply(label_sentiment)
df = df[["reviews.text", "label"]].reset_index(drop=True)

print("Class distribution:")
print(df["label"].value_counts(normalize=True).rename("ratio"))
print(f"Total cleaned samples: {len(df)}")

Class distribution:
label
1    0.843612
0    0.156388
Name: ratio, dtype: float64
Total cleaned samples: 908


In [9]:
df.sample(5, random_state=SEED)

,reviews.text,label
865,Originally got the walnut cover with my Oasis....,1
439,"Bought the Echo Tap, Echo Dot, Phillips Hue St...",1
342,I habe the echo and I bought this one fort my ...,0
736,Much more than a Bluetooth speaker. It is grea...,1
785,"I bring it when hanging out outside, exercisin...",1


In [10]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED,
)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 726
Test size: 182


## 2. Tokenization

In [11]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["reviews.text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

## 3. Build Hugging Face Datasets

In [12]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print(train_dataset)
print(test_dataset)

Map: 100%|██████████| 182/182 [00:00<00:00, 2273.99 examples/s]

Dataset({
    features: ['reviews.text', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 726
})
Dataset({
    features: ['reviews.text', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 182
})


## 4. Handle Class Imbalance with Weighted Loss

In [ ]:
from torch.cuda import device


labels = train_df["label"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels,
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class weights on {device}: {class_weights}")

NameError: name 'device' is not defined

In [14]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)
model.to(device)

d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devdp\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2174.07it/s]
DistilBertForSequenceCl

NameError: name 'device' is not defined

### Why a Custom Trainer?

The default Trainer uses standard cross-entropy and may underperform on imbalanced labels.
This custom Trainer overrides the loss function so minority-class errors receive higher penalty,
which improves recall and macro-F1 stability.

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels") if "labels" in inputs else inputs.get("label")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_strategy="epoch",
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [123]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.494757
2,No log,0.389537
3,No log,0.419098
4,No log,0.395089


TrainOutput(global_step=236, training_loss=0.35774848420741195, metrics={'train_runtime': 114.0383, 'train_samples_per_second': 33.006, 'train_steps_per_second': 2.069, 'total_flos': 498607288541184.0, 'train_loss': 0.35774848420741195, 'epoch': 4.0})

In [ ]:
eval_metrics = trainer.evaluate()
eval_metrics

{'eval_loss': 0.3895372152328491,
 'eval_runtime': 3.3111,
 'eval_samples_per_second': 71.276,
 'eval_steps_per_second': 2.416,
 'epoch': 4.0}

In [ ]:
# Predictions
train_preds = trainer.predict(train_dataset)
test_preds = trainer.predict(test_dataset)

train_pred_labels = np.argmax(train_preds.predictions, axis=1)
test_pred_labels = np.argmax(test_preds.predictions, axis=1)

train_true = train_preds.label_ids
test_true = test_preds.label_ids

label_names = ["negative", "positive"]

print("TRAIN REPORT")
print(classification_report(train_true, train_pred_labels, target_names=label_names))

print("\nTEST REPORT")
print(classification_report(test_true, test_pred_labels, target_names=label_names))

cm = confusion_matrix(test_true, test_pred_labels)
print("\nTest Confusion Matrix:")
print(cm)

🔹 TRAIN REPORT
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86       160
    positive       0.98      0.97      0.97       781

    accuracy                           0.95       941
   macro avg       0.91      0.92      0.92       941
weighted avg       0.95      0.95      0.95       941


🔹 TEST REPORT
              precision    recall  f1-score   support

    negative       0.74      0.78      0.76        40
    positive       0.95      0.94      0.95       196

    accuracy                           0.92       236
   macro avg       0.85      0.86      0.85       236
weighted avg       0.92      0.92      0.92       236



In [ ]:
def predict(text: str):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    labels = ["negative", "positive"]

    return {
        "label": labels[pred_idx],
        "confidence": float(probs[pred_idx]),
        "scores": {labels[i]: float(probs[i]) for i in range(len(labels))},
    }

## 5. Inference Examples

In [ ]:
predict("This product is amazing but delivery was slow")

negative


In [ ]:
predict("This is the worst product I have ever bought")

negative


In [ ]:
predict("The product is okay, nothing special")

positive


In [ ]:
predict("The product is good but delivery was very slow")

negative


## 6. Improvement Ideas

1. Compare DistilBERT with RoBERTa-base or DeBERTa-v3-small.
2. Add early stopping and hyperparameter search (learning rate, batch size, epochs).
3. Track experiments with Weights & Biases or MLflow.
4. Add text cleaning/normalization ablations to quantify preprocessing impact.
5. Build a lightweight API endpoint for real-time sentiment inference.

### Suggested Next Training Configuration

```python
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.05,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)
```

## 7. Save Model Artifacts

In [ ]:
import json

artifact_dir = Path("sentiment_model")
artifact_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(artifact_dir)
tokenizer.save_pretrained(artifact_dir)

label_mapping = {0: "negative", 1: "positive"}
with open(artifact_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2)

with open(artifact_dir / "eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

print(f"Artifacts saved to: {artifact_dir.resolve()}")

('sentiment_model\\tokenizer_config.json',
 'sentiment_model\\special_tokens_map.json',
 'sentiment_model\\vocab.txt',
 'sentiment_model\\added_tokens.json')

In [ ]:
# Quick reload test for deployment-style inference
loaded_tokenizer = DistilBertTokenizer.from_pretrained("sentiment_model")
loaded_model = DistilBertForSequenceClassification.from_pretrained("sentiment_model").to(device)
loaded_model.eval()

sample_text = "Excellent quality and fast delivery"
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True).to(device)

with torch.no_grad():
    logits = loaded_model(**inputs).logits

pred = int(torch.argmax(logits, dim=1).item())
print({0: "negative", 1: "positive"}[pred])